<a href="https://colab.research.google.com/github/Innovatewithapple/LangChain-Basic/blob/main/PaperRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-huggingface langchain_community pymupdf faiss-cpu

In [ ]:
#Documents Loader
from langchain_community.document_loaders import PyMuPDFLoader
import pymupdf
import re
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from langchain_core.documents import Document
import torch
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer,AutoModelForCausalLM
from langchain_core.embeddings import Embeddings
from typing import List
import faiss
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('huggingfaceToken'))

In [ ]:
#----------Load Encoder Model------------#
encoder_Tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")
encoder_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#---------Load Decoder Models-----------#
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token  # ← add this
llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

# ---- LOAD RERANKER (once at startup) ----
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:
def load_Process(path):
  doc = pymupdf.open(path)
  pages = []
  for page_no,page in enumerate(doc):
    page = page.get_text()
    clean_text = re.sub(r'\n{3,}', '\n\n', page) #Removes excessive empty lines.
    clean_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', clean_text) #This fixes broken sentences. next char or previous char shouldn't be in next line
    clean_text = re.sub(r' {2,}', ' ', clean_text) #Removes multiple spaces.
    pages.append({
        'page_no':page_no,
        'page':clean_text
    })

  return pages

files = load_Process('/content/NIPS-2017-attention-is-all-you-need-Paper.pdf')

In [ ]:
len(files)

In [ ]:
def Create_parent_child_chunks(pages,child_token_size,child_overlap_size,parent_token_size,parent_overlap_size):
  results = []
  for page in pages:
    parent_chunks = []
    parent_current_chunk=[]
    parent_current_len_list = []
    parent_current_len=0

    sentence = sent_tokenize(text=page['page'])
    parent_token_len = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentence]

    for parent_sent,parent_token in zip(sentence,parent_token_len):
      if parent_current_len + parent_token < parent_token_size:
        parent_current_chunk.append(parent_sent)
        parent_current_len_list.append(parent_token)
        parent_current_len += parent_token
      else:
        if parent_current_chunk:
          parent_chunks.append(" ".join(parent_current_chunk))
        parent_current_chunk = parent_current_chunk[-parent_overlap_size:] + [parent_sent]
        parent_current_len_list = parent_current_len_list[-parent_overlap_size:] + [parent_token]
        parent_current_len = sum(parent_current_len_list)

    if parent_current_chunk:
      parent_chunks.append(" ".join(parent_current_chunk))

    #-------Child Chunks-------!
    for p_idx,parent_chunk in enumerate(parent_chunks):
      parent_sentence = sent_tokenize(parent_chunk)
      parent_chunk_total_len = [len(encoder_Tokenizer.encode(parent_sentence,add_special_tokens=False)) for sent in parent_sentence]

      child_chunks = []
      child_current_chunk=[]
      child_current_len_list=[]
      child_current_len=0

      for parent_sent,parent_token in zip(parent_sentence,parent_chunk_total_len):
        if child_current_len + parent_token < child_token_size:
          child_current_chunk.append(parent_sent)
          child_current_len_list.append(parent_token)
          child_current_len += parent_token
        else:
          if child_current_chunk:
            child_chunks.append(" ".join(child_current_chunk))
          child_current_chunk = child_current_chunk[-child_overlap_size:] + [parent_sent]
          child_current_len_list = child_current_len_list[-child_overlap_size:] + [parent_token]
          child_current_len = sum(child_current_len_list)

      if child_current_chunk:
        child_chunks.append(" ".join(child_current_chunk))

      #-----Result----!
      for child_idx,child in enumerate(child_chunks):
          results.append(
              Document(
                  page_content=child,
                  metadata={
                  'page_no':page['page_no'],
                  'parent_text':parent_chunk,
                  'parent_idx':p_idx,
                  'child_id':f'{p_idx}_{child_idx}'
                  }
                  ))
  return results

In [ ]:
all_chunks = Create_parent_child_chunks(pages=files,child_token_size=300,child_overlap_size=3,parent_token_size=600,parent_overlap_size=5)

In [ ]:
class PreloadSentenceTransformerEmbeddings(Embeddings):
  def __init__(self,model):
    self.model = model

  def embed_documents(self,texts: List[str]) -> List[List[float]]:
    return self.model.encode(texts,normalize_embeddings=True,batch_size=32,show_progress_bar=True).tolist()

  def embed_query(self,text:str) -> List[float]:
    return self.model.encode(text,normalize_embeddings=True).tolist()

embeddings = PreloadSentenceTransformerEmbeddings(encoder_model)

In [ ]:
vector_stores = FAISS.from_documents(documents=all_chunks,embedding=embeddings)

In [ ]:
retriever = vector_stores.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [ ]:
reranker = RunnableLambda(LangChainReranker(reranking_model,top_k=3))

In [ ]:
parser = StrOutputParser()

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a precise question-answering assistant.

        Answer using ONLY the provided context.

        Rules:
        1. Do NOT use outside knowledge.
        2. Copy numerical and decimal values exactly as written in the context.
        3. Do NOT round, estimate, modify, or infer numbers.
        4. Do NOT explain or expand abbreviations unless explicitly written in the context.
        5. Preserve important entities, task names, benchmark names, and relationships exactly as written in the context.
        6. Include the specific task or subject associated with the answer if present in the context.
        7. Give a clear and natural answer in 1-2 sentences.
        8. Keep the answer grounded strictly in the provided context.
        9. If the exact answer is not explicitly stated but can be reasonably inferred from the context, provide the most likely answer based ONLY on the context.
        10.If the context does not contain enough information, say:
           "I don't know."
        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}
        """
    )
])

In [ ]:
pipe = pipeline('text-generation',
                model=llm,
                tokenizer=decoder_tokenizer,
                max_new_tokens=80,
                max_length=None,
                temperature=0.1,
                do_sample=True,
                repetition_penalty=1.0,
                no_repeat_ngram_size=3,
                top_p=0.9,
                pad_token_id=decoder_tokenizer.eos_token_id,
                return_full_text=False)

llm_pipeline = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm_pipeline)

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | reranker
    | prompt
    | chat_model
    | parser
)

In [ ]:
answer = rag_chain.invoke("What BLEU score achieved in English-German translation?")
answer

In [ ]:
class LangChainReranker:
  def __init__(self,model,top_k=3):
    self.model = model
    self.top_k = top_k

  def __call__(self,inputs):
    question = inputs['question']
    docs = inputs['context']

    pairs = [(question,doc.page_content) for doc in docs]
    scores = self.model.predict(inputs=pairs)

    for i,doc in enumerate(docs):
      doc.metadata['rerank_score'] = scores[i]

    reranked = sorted(docs,key=lambda x:x.metadata['rerank_score'],reverse=True)
    top_docs = reranked[:self.top_k]

    unique_parents = []

    seen = set()

    for doc in top_docs:

      parent_id = doc.metadata["parent_idx"]

      if parent_id not in seen:
        seen.add(parent_id)
        unique_parents.append(
            doc.metadata["parent_text"]
        )

    return {
    "question": question,
    "context": "\n\n".join(unique_parents)
    }

    # return {
    #     'question':question,
    #     'context':"\n\n".join(doc.page_content for doc in docs)
    # }
